# 🔺 Delta Lake — Liquid Clustering

> *"Empty your mind. Be formless, shapeless — like water."*  
> — Bruce Lee (inspiração para o Liquid Clustering)

---

## 1. O que é Liquid Clustering?

O Liquid Clustering é a evolução do Z-Ordering.  
Ele mantém os benefícios da co-localização de dados, mas resolve as principais limitações do Z-Order e do Partitioning.
![Foto](./Fotos/20.png)
![Foto](./Fotos/21.png)
![Foto](./Fotos/22.png)
---

### O problema com Z-Ordering e Partitioning

```
  PARTITIONING
  ─────────────────────────────────────────────────────────────────
  ❌ Estrutura gravada em pedra — mudar a coluna exige reescrever
     TODA a tabela
  ❌ Data skewness: distribuição desigual entre subdiretórios
  ❌ Over-partitioning: small files se a coluna tiver alta cardinalidade
  ❌ Benefício não é sempre óbvio vs File Skipping do _delta_log

  Z-ORDERING
  ─────────────────────────────────────────────────────────────────
  ❌ Cada OPTIMIZE reescreve TODOS os dados — custo alto em escala
  ❌ Não é visível no DESC DETAIL — difícil de auditar
  ❌ Não pode ser definido na criação da tabela
  ❌ Não é incremental — precisa re-rodar manualmente sempre
  ❌ Algoritmo Z tem problem: cria arquivos com min/max muito amplo
     em uma das colunas → arquivo impossível de ser skipado
```

---

### Hilbert Curve — algoritmo mais preciso que o Z

O Liquid Clustering substitui a Curva Z pela **Curva de Hilbert**.  
Ambas percorrem um espaço multidimensional para co-localizar linhas similares,  
mas a Hilbert Curve produz arquivos com intervalos estatísticos muito mais estreitos:

```
  Z-ORDERING (Curva Z)              LIQUID CLUSTERING (Curva de Hilbert)
  ──────────────────────────        ──────────────────────────────────────
  Arquivo problemático (azul):      Todos os arquivos:
  ColA: min=1, max=4                ColA: intervalo estreito
  ColB: min=1, max=8  ← amplo!      ColB: intervalo estreito

  → Esse arquivo nunca pode ser     → Todos os arquivos podem ser
    skipado em queries sobre ColB     skipados com precisão
    independente do filtro

  Z-Order melhora sobre Partitioning.
  Hilbert Curve melhora sobre Z-Order.
```

> **Por que isso importa?**  
> Arquivos com intervalos estatísticos amplos no `_delta_log` não podem ser eliminados pelo File Skipping.  
> O Spark é forçado a lê-los independente do filtro da query — mesmo que o dado não esteja ali.

---

### O que há de novo no Liquid Clustering

| Limitação | Partitioning | Z-Ordering | Liquid Clustering |
|---|---|---|---|
| Visível no `DESC DETAIL` | ✅ | ❌ | ✅ |
| Definido na criação da tabela | ✅ | ❌ | ✅ |
| Coluna alterável sem reescrever tudo | ❌ | ❌ | ✅ |
| Clustering incremental (sem OPTIMIZE) | ❌ | ❌ | ✅ (às vezes) |
| Reescreve apenas dados não-estáveis | ❌ | ❌ | ✅ |
| Subdiretórios físicos | ✅ | ❌ | ❌ |
| Algoritmo | — | Curva Z | Curva de Hilbert |

---

### O conceito de "Stable Size" — o que faz o clustering ser *liquid*

```
  Arquivo pequeno / novo              Arquivo grande / estável
  ("liquid" — ainda fluido)           ("stable size" atingido)
          │                                    │
          ▼                                    ▼
  OPTIMIZE o reclusteriza             OPTIMIZE NÃO toca
  Hilbert Curve reorganiza            Arquivo permanece intacto
  Arquivos crescem até stable size    Mesmo mudando as colunas
                                      de clustering!

  → Diferença fundamental vs Z-Order:
    Z-Order reescreve TUDO a cada OPTIMIZE
    Liquid Clustering reescreve apenas o que ainda não é estável
```

> Quando muitos dados são deletados, arquivos que eram estáveis podem se tornar  
> "liquid" novamente e serão reclustrerizados na próxima otimização.

---

## 2. Criando a tabela com Clustering

Diferente do Z-Ordering, o Liquid Clustering é definido **na criação da tabela** com `CLUSTER BY`.  
Isso já sinaliza ao Delta Lake para aplicar a Hilbert Curve durante os writes.

In [ ]:
%%sql
DROP TABLE IF EXISTS flight_data_lc;

CREATE TABLE flight_data_lc (
    Year INT, Month INT, DayofMonth INT, DayOfWeek INT, DepTime STRING, CRSDepTime INT, ArrTime STRING, CRSArrTime INT, UniqueCarrier STRING, FlightNum INT, TailNum STRING, ActualElapsedTime STRING, CRSElapsedTime STRING, AirTime STRING, ArrDelay STRING, DepDelay STRING, Origin STRING, Dest STRING, Distance INT, TaxiIn STRING, TaxiOut STRING, Cancelled INT, CancellationCode STRING, Diverted INT, CarrierDelay STRING, WeatherDelay STRING, NASDelay STRING, SecurityDelay STRING, LateAircraftDelay STRING
)
CLUSTER BY (Year, Dest);

In [ ]:
%python
spark.read.csv("dbfs:/databricks-datasets/asa/airlines/2*.csv", header=True, inferSchema=True)\
    .write.mode("append").saveAsTable("flight_data_lc")

---
## 3. Inspecionando a tabela

### 3.1 — `DESC DETAIL` mostra as colunas de clustering diretamente

In [ ]:
%%sql
-- Diferente do Z-Ordering e Bloom Filter, aqui a coluna 'clusteringColumns'
-- aparece diretamente — não é necessário ir ao histórico para saber
DESC DETAIL flight_data_lc;

> **Vantagem sobre Z-Ordering:**  
> Com Z-Ordering, a única forma de saber quais colunas foram ordenadas era consultar o `DESC HISTORY`.  
> Com Liquid Clustering, as colunas de clustering são parte da **definição da tabela** — visíveis instantaneamente.

### 3.2 — `DESC HISTORY` mostra o clustering incremental

In [ ]:
%%sql
-- Procure em 'operationParameters' a chave 'clusteringOnWriteStatus'
-- Quando o clustering incremental ocorre, essa chave aparece com status de sucesso
DESC HISTORY flight_data_lc;

> **O que é o clustering incremental?**  
> Em alguns writes, o Delta Lake aplica a Hilbert Curve automaticamente durante a ingestão,  
> sem precisar rodar `OPTIMIZE` separadamente.  
> Quando isso ocorre, `clusteringOnWriteStatus` aparece nos `operationParameters` do write.

### 3.3 — Verificando a estrutura de arquivos

In [ ]:
-- Arquivos menores que o esperado — o clustering ainda não atingiu 'stable size'
-- Os dados estão 'liquid': prontos para serem reclustrerizados e compactados
%fs
ls dbfs:/user/hive/warehouse/flight_data_lc

> **Por que os arquivos são menores que no Z-Ordering?**  
> No Z-Ordering, todos os dados eram compactados de uma vez em poucos arquivos grandes.  
> No Liquid Clustering, os arquivos iniciais ainda são "liquid" — o clustering final  
> (com Hilbert Curve completa e arquivos na stable size) ocorre progressivamente.

### 3.4 — OPTIMIZE logo após a ingestão

In [ ]:
%%sql
-- Se o clustering incremental já foi feito no write:
-- numFilesAdded = 0, numFilesRemoved = 0
-- O Liquid Clustering reconhece que não há o que otimizar
OPTIMIZE flight_data_lc;

> Se o OPTIMIZE retornou `numFilesAdded = 0` e `numFilesRemoved = 0`,  
> significa que o clustering incremental foi executado durante a ingestão — nada a fazer.

---

## 4. Comportamento inconsistente do clustering incremental

O clustering incremental é uma feature poderosa, mas **não é garantido**.  
Vamos demonstrar dois cenários onde ele não dispara automaticamente.

### 4.1 — Insert pequeno (100 linhas)

In [ ]:
%%sql
INSERT INTO flight_data_lc
SELECT *
FROM flight_data_lc
LIMIT 100;

In [ ]:
%%sql
-- Procure em 'operationParameters' → 'clusteringOnWriteStatus'
-- Mensagem esperada: "Reason for skipping: Cannot estimate ingestion size"
-- O Delta Lake não consegue estimar o tamanho para decidir se vale clusterizar
DESC HISTORY flight_data_lc;

> **`Cannot estimate ingestion size`:**  
> O clustering incremental foi pulado porque o Delta Lake não conseguiu estimar  
> o tamanho da ingestão para decidir se o clustering automático seria eficiente.  
> Isso é inconsistente — às vezes funciona, às vezes não.

### 4.2 — Insert grande (5x o volume de dados)

In [ ]:
%%sql
-- Multiplica o volume da tabela por ~5 via UNION (sem UNION ALL — deduplica)
INSERT INTO flight_data_lc
SELECT *
FROM flight_data_lc
UNION
SELECT *
FROM flight_data_lc
UNION
SELECT *
FROM flight_data_lc;

In [ ]:
%%sql
-- Mesmo com um volume muito maior, a mensagem de skip pode aparecer novamente
-- → Conclusão: não podemos depender do clustering incremental
DESC HISTORY flight_data_lc;

> ⚠️ **Lição importante:**  
> O clustering incremental é uma otimização *best effort* — não é garantido nem previsível.  
> **Não dependa dele em pipelines de produção.**  
> Sempre execute `OPTIMIZE` periodicamente, como faria com Z-Ordering.

---

## 5. Executando OPTIMIZE manualmente

Após os inserts, a tabela acumulou muitos arquivos pequenos.  
O OPTIMIZE aplica a Hilbert Curve e compacta os arquivos liquid em arquivos maiores.

In [ ]:
-- Muitos arquivos pequenos antes da otimização
%fs
ls dbfs:/user/hive/warehouse/flight_data_lc

In [ ]:
%%sql
-- OPTIMIZE sem ZORDER BY — o Liquid Clustering usa automaticamente a Hilbert Curve
-- Resultado esperado: ~200 filesRemoved, ~8 filesAdded
OPTIMIZE flight_data_lc;

In [ ]:
%%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM flight_data_lc RETAIN 0 HOURS;

In [ ]:
-- Poucos arquivos grandes (~189 MB cada) — stable size atingido
-- Esses arquivos NÃO serão reescritos em otimizações futuras
%fs
ls dbfs:/user/hive/warehouse/flight_data_lc

> **Comparação com Z-Ordering:**  
> Z-Ordering gerava arquivos de ~300 MB após OPTIMIZE.  
> Liquid Clustering gerou ~189 MB — arquivos menores, mas com clustering mais preciso  
> (Hilbert Curve tem intervalos estatísticos mais estreitos → melhor skip de arquivos).
>
> **O que é "stable size"?**  
> Arquivos que atingiram o tamanho alvo do Liquid Clustering.  
> A partir desse ponto, o OPTIMIZE **não os toca mais** — mesmo mudando as colunas de clustering.

---

## 6. Alterando as colunas de clustering

Esta é uma das vantagens mais importantes do Liquid Clustering sobre o Partitioning:  
**é possível mudar a estratégia de clustering sem reescrever toda a tabela**.

### 6.1 — Mudando para clustering por Month

In [ ]:
%%sql
-- Altera as colunas de clustering: (Year, Dest) → (Month)
-- ⚠️ Os arquivos já estáveis (stable size) NÃO serão reescritos
-- Apenas novos dados e arquivos liquid serão reclustrerizados com Month
ALTER TABLE flight_data_lc CLUSTER BY (Month);

In [ ]:
%%sql
OPTIMIZE flight_data_lc;

In [ ]:
%%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM flight_data_lc RETAIN 0 HOURS;

In [ ]:
-- Número de arquivos diminuiu, tamanho aumentou (~240 MB nos maiores)
-- PORÉM: dois arquivos são idênticos aos do clustering anterior (Year, Dest)
-- → Esses eram estáveis e NÃO foram reescritos mesmo mudando a coluna
%fs
ls dbfs:/user/hive/warehouse/flight_data_lc

> **O que aconteceu:**  
> Arquivos que já tinham atingido stable size com `(Year, Dest)` foram mantidos intactos.  
> Apenas os arquivos liquid (pequenos, recentes) foram reclustrerizados com `Month`.  
>
> Compare com o que aconteceria no Z-Ordering ou Partitioning:
> ```
>   Z-Ordering:    OPTIMIZE reescreve TODOS os arquivos → custo total
>   Partitioning:  Precisa recriar a tabela inteira → inviável em produção
>   Liquid Clust.: Reescreve apenas o que é liquid → custo mínimo
> ```

### 6.2 — Tentando mudar novamente (UniqueCarrier, TailNum)

In [ ]:
%%sql
ALTER TABLE flight_data_lc CLUSTER BY (UniqueCarrier, TailNum);
OPTIMIZE flight_data_lc;

> **Resultado esperado:** OPTIMIZE pode retornar `numFilesAdded = 0, numFilesRemoved = 0`.  
> Isso significa que todos os arquivos já atingiram stable size e o Liquid Clustering  
> decidiu não tocá-los — mesmo com uma coluna diferente sendo definida.  
> O clustering só será aplicado para **novos dados** que chegarem a partir de agora.

---

## 7. Removendo o Clustering

In [ ]:
%%sql
-- Remove completamente o clustering da tabela
-- Dados existentes não são alterados — apenas novos writes não serão clusterizados
ALTER TABLE flight_data_lc CLUSTER BY NONE;

---
## 8. Resumo — Liquid Clustering

```
┌──────────────────────────────────────────────────────────────────┐
│              LIQUID CLUSTERING — CICLO DE VIDA                   │
│                                                                  │
│  CREATE TABLE ... CLUSTER BY (Year, Dest)                        │
│        │                                                         │
│  INSERT (ingestão)                                               │
│        ├── Clustering incremental: às vezes ocorre, às vezes não │
│        └── Arquivos gerados são "liquid" (pequenos, instáveis)   │
│                                                                  │
│  OPTIMIZE                                                        │
│        ├── Aplica Hilbert Curve nos arquivos liquid              │
│        ├── Compacta em arquivos maiores                          │
│        └── Quando atingem stable size → não serão tocados mais   │
│                                                                  │
│  ALTER TABLE ... CLUSTER BY (Month)                              │
│        ├── Arquivos stable: MANTIDOS INTACTOS                    │
│        └── Arquivos liquid + novos dados: reclustrerizados        │
│                                                                  │
│  ALTER TABLE ... CLUSTER BY NONE                                 │
│        └── Remove clustering — dados existentes inalterados      │
└──────────────────────────────────────────────────────────────────┘
```

### Comparação final — todas as estratégias

| | File Skipping | Partitioning | Z-Ordering | Liquid Clustering |
|---|---|---|---|---|
| Automático | ✅ | ❌ | ❌ | ⚡ Parcial |
| Visível no `DESC DETAIL` | ❌ | ✅ | ❌ | ✅ |
| Definido na criação | ❌ | ✅ | ❌ | ✅ |
| Coluna alterável | ❌ | ❌ (reescreve tudo) | ❌ (reescreve tudo) | ✅ |
| Reescreve apenas dados instáveis | ❌ | ❌ | ❌ | ✅ |
| Algoritmo de co-localização | Stats min/max | Subdiretórios | Curva Z | Curva de Hilbert |
| Recomendação Databricks | Sempre ativo | > 1TB, baixa cardinalidade | Alta cardinalidade | **Substitui ambos** |

### Comandos de referência

| Objetivo | Comando |
|---|---|
| Criar tabela com clustering | `CREATE TABLE t (...) CLUSTER BY (col1, col2)` |
| Ver colunas de clustering | `DESC DETAIL t` → coluna `clusteringColumns` |
| Verificar status incremental | `DESC HISTORY t` → `operationParameters.clusteringOnWriteStatus` |
| Executar clustering manual | `OPTIMIZE t` |
| Alterar colunas de clustering | `ALTER TABLE t CLUSTER BY (nova_col)` |
| Remover clustering | `ALTER TABLE t CLUSTER BY NONE` |